# PyGeoFetch — 03: Advanced Search

Multi-provider federated search, CQL2 filters, SLC product routing,
Sentinel-1C/D, and all 7 output formats.

### What you'll learn
- Federated search across 3 free providers
- CQL2 filter expressions
- SLC product type and automatic provider routing (Capability 1)
- Sentinel-1C scene discovery (Capability 2)
- All 7 output formats: table, JSON, STAC, GeoJSON, GeoParquet, CSV, IDs
- Cache management

In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pathlib import Path
import json, pandas as pd

client = PyGeoFetch(log_level="INFO")
Path("output").mkdir(exist_ok=True)
nyc_bbox = BoundingBox.from_string("-74.1,40.6,-73.7,40.9")

## 1. Federated Search — 3 Free Providers

In [ ]:
# The new log format matches the spec:
# 10:42:15 INFO [    search] Searching 3 provider(s) for all scenes
# 10:42:15 INFO [    search]   bbox       : [-74.1000, 40.6000, -73.7000, 40.9000]
# 10:42:15 INFO [    search]   date range : 2024-01-01 -> 2024-06-01
# ...

query = SearchQuery(
    bbox=nyc_bbox,
    start_date="2024-01-01",
    end_date="2024-06-01",
    cloud_cover_max=20,
    max_results=50,
    sort_by="cloud_cover",
    sort_ascending=True,
)

results = client.search(
    query,
    providers=["aws_earth", "planetary_computer", "element84"]
)
print(f"Found {len(results)} unique scenes (deduplicated)")

## 2. Satellite and Product Type Filters

In [ ]:
# Sentinel-2 only, sorted by cloud cover
q_s2 = SearchQuery(
    bbox=nyc_bbox,
    start_date="2024-01-01",
    end_date="2024-06-01",
    cloud_cover_max=10,
    satellites=["Sentinel-2"],
    sort_by="cloud_cover",
    sort_ascending=True,
)
results_s2 = client.search(q_s2, providers=["aws_earth"])
print(f"Sentinel-2 results: {len(results_s2)}")
for r in results_s2[:5]:
    print(f"  {(r.satellite or "—"):<15} {str(r.datetime)[:10]}  cloud={r.cloud_cover:.1f}%")

## 3. SLC Product Type — Capability 1

In [ ]:
# SLC (Single Look Complex) is needed for InSAR mm-precision deformation mapping.
# SearchQuery now has a product_type field and set_product_type() builder.

q_slc = SearchQuery(
    bbox=BoundingBox.from_string("-0.3,5.4,0.2,5.9"),  # Ghana
    start_date="2026-06-01",
    end_date="2026-06-30",
)
# Builder method — sets product_type and uppercases
q_slc.set_product_type("SLC")

print(f"product_type = {q_slc.product_type!r}")

# Provider routing: planetary_computer and element84 only have GRD.
# PyGeoFetch automatically routes SLC queries to capable providers.
print()
print("SLC provider routing:")
grd_only_providers = ["planetary_computer", "aws_earth", "element84"]
routed = client._route_slc_providers(grd_only_providers)
print(f"  Requested : {grd_only_providers}")
print(f"  Routed to : {routed}")
print()
print("SLC-capable providers: copernicus, alaska_satellite_facility, asf_vertex")

# Note: actual SLC download requires Copernicus credentials
# results_slc = client.search(q_slc, providers=["copernicus"])

## 4. Sentinel-1C Scene Discovery — Capability 2

In [ ]:
# Search for Sentinel-1C/D scenes
# After July 2026, all S1 results should come from S1C or S1D
from datetime import date

q_s1c = SearchQuery(
    bbox=BoundingBox.from_string("-0.3,5.4,0.2,5.9"),
    start_date="2026-06-01",
    end_date="2026-06-30",
)

# Platforms searched: SENTINEL-1A, SENTINEL-1B, SENTINEL-1C, SENTINEL-1D
# (all 4 now in S1_PLATFORMS for every SAR provider)

print("Provider platform list now includes S1C and S1D:")
print("  S1_PLATFORMS = ['SENTINEL-1A','SENTINEL-1B','SENTINEL-1C','SENTINEL-1D']")
print()
print("_warn_if_outdated_constellation() fires if only S1A/S1B returned after July 2026")
print(f"Today: {date.today()} -- decommissioning threshold: 2026-07-01")

# Live search (uncomment with real credentials):
# results_s1c = client.search(q_s1c, providers=["copernicus"])
# for r in results_s1c[:5]:
#     print(f"  {r.satellite}  {r.polarisation}  {r.pass_direction}  orbit={r.relative_orbit}")

## 5. CQL2 Filter Expressions

In [ ]:
# CQL2 is the OGC filter language for STAC APIs
# Supported by: planetary_computer, element84, aws_earth

q_cql2 = SearchQuery(
    bbox=nyc_bbox,
    start_date="2024-01-01",
    end_date="2024-06-01",
    cql2_filter="eo:cloud_cover < 5 AND platform = 'sentinel-2b'",
)

print("CQL2 filter examples:")
examples = [
    ("Cloud filter",        "eo:cloud_cover < 10"),
    ("Platform filter",     "platform = 'sentinel-2b'"),
    ("Combined",            "eo:cloud_cover < 5 AND platform = 'sentinel-2b'"),
    ("Processing level",    "s2:processing_level = 'L2A'"),
    ("Landsat tier",        "landsat:collection_category = 'T1'"),
]
for name, expr in examples:
    print(f"  [{name}]  --cql2 "{expr}"")

## 6. All 7 Output Formats

In [ ]:
sample = results[:10] if results else []

# Format 1: JSON
json_records = [
    {"id": r.id, "provider": r.provider, "satellite": r.satellite,
     "date": str(r.datetime)[:10] if r.datetime else None,
     "cloud_pct": r.cloud_cover,
     "product_type": r.product_type}
    for r in sample
]
json_path = Path("output/results.json")
json_path.write_text(json.dumps(json_records, indent=2))
print(f"1. JSON    -> {json_path} ({len(sample)} records)")

# Format 2: STAC FeatureCollection
stac_fc = client.searcher.to_geojson(sample)
stac_path = Path("output/results_stac.json")
with open(stac_path, "w") as f:
    json.dump(stac_fc, f, indent=2, default=str)
print(f"2. STAC    -> {stac_path}")

# Format 3: GeoJSON
geojson_path = Path("output/results.geojson")
client.searcher.save_results(sample, geojson_path)
print(f"3. GeoJSON -> {geojson_path}")

# Format 4: CSV
import csv
csv_path = Path("output/results.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["id","provider","satellite","date","cloud_pct","product_type"])
    w.writeheader()
    w.writerows(json_records)
print(f"4. CSV     -> {csv_path}")

# Format 5: GeoParquet (requires geopandas)
try:
    import geopandas as gpd
    from shapely.geometry import box as shapely_box
    geometries = []
    for r in sample:
        if r.bbox:
            geometries.append(shapely_box(r.bbox.min_lon, r.bbox.min_lat,
                                          r.bbox.max_lon, r.bbox.max_lat))
        else:
            geometries.append(None)
    gdf = gpd.GeoDataFrame(json_records, geometry=geometries, crs="EPSG:4326")
    gdf.to_parquet("output/results.geoparquet")
    print(f"5. GeoParquet -> output/results.geoparquet")
except ImportError:
    print("5. GeoParquet -> pip install geopandas to enable")

# Format 6: IDs only
ids_path = Path("output/scene_ids.txt")
ids_path.write_text("
".join(r.id for r in sample))
print(f"6. IDs     -> {ids_path}")

# Format 7: CLI table (handled by the CLI automatically)
print("7. Table   -> PyGeoFetch search run ... --format table")

## 7. Cache Management

In [ ]:
import time

q_cache = SearchQuery(bbox=nyc_bbox, start_date="2024-01-01", end_date="2024-03-01",
                      cloud_cover_max=10)

t0 = time.time()
r1 = client.search(q_cache, providers=["aws_earth"])
t1 = time.time()
r2 = client.search(q_cache, providers=["aws_earth"])  # cache hit
t2 = time.time()

print(f"First search:  {t1-t0:.2f}s  ({len(r1)} results)")
print(f"Cached search: {t2-t1:.4f}s  ({len(r2)} results)  — instant!")
print()

# Clear cache
cleared = client.clear_cache()
print(f"Cleared {cleared} cache entries")

---
## Summary
- Federated search with INFO-level log output in spec format
- product_type field + set_product_type() builder on SearchQuery
- SLC routing: GRD-only providers auto-routed to copernicus for SLC queries
- Sentinel-1C/D in all provider platform lists
- CQL2 filter expressions for STAC APIs
- All 7 output formats: JSON, STAC, GeoJSON, GeoParquet, CSV, IDs, table

**Next:** Notebook 04 — Download, Validation & Post-Processing